## EdTech Content Data Pipeline
### AI · Data · Cloud — RSS Feeds Collection

###  Objective
Build a **repeatable Data Engineering pipeline** that collects, standardizes, enriches, and curates educational content about **AI**, **Data**, and **Cloud** from public RSS sources.




### 1. Install and import the libraries

In [0]:
# Install and import required libraries

%pip install feedparser

import feedparser
import hashlib
import json
import re
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### 2. Define the sources of blog and newsletter RSS feeds 

In [0]:
BLOG_FEEDS = {
    # AI
    "OpenAI": "https://openai.com/news/rss.xml",
    "Google AI": "https://blog.google/technology/ai/rss/",
    "Google DeepMind": "https://deepmind.google/blog/rss.xml",
    "Google Research": "https://research.google/blog/rss/",
    "Hugging Face": "https://huggingface.co/blog/feed.xml",
    "Meta Engineering": "https://engineering.fb.com/feed/",
    "Apple Machine Learning": "https://machinelearning.apple.com/rss.xml",
    "NVIDIA": "https://blogs.nvidia.com/feed/",
    "NVIDIA Developer": "https://developer.nvidia.com/blog/feed/",
    "Simon Willison": "https://simonwillison.net/atom/everything/",
    "TechCrunch AI": "https://techcrunch.com/category/artificial-intelligence/feed/",
    "MIT Technology Review AI": "https://www.technologyreview.com/topic/artificial-intelligence/feed/",
    "The Decoder": "https://the-decoder.com/feed/",
    "The Verge AI": "https://www.theverge.com/rss/ai-artificial-intelligence/index.xml",

    # Data
    "Netflix Tech Blog": "https://netflixtechblog.com/feed",
    "Spotify Engineering": "https://engineering.atspotify.com/feed/",
    "InfoQ Data": "https://feed.infoq.com/data/",
    "InfoQ Big Data": "https://feed.infoq.com/bigdata/",
    "InfoQ Data Engineering": "https://feed.infoq.com/ai-ml-data-eng/",

    # Cloud
    "Cloudflare": "https://blog.cloudflare.com/rss/",
    "Kubernetes": "https://kubernetes.io/feed.xml",
    "AWS Compute": "https://aws.amazon.com/blogs/compute/feed/",
    "AWS News": "https://aws.amazon.com/blogs/aws/feed/"
}

In [0]:
SUBSTACK_FEEDS = {
    "Stratechery": "https://stratechery.com/feed",
    "Astral Codex Ten": "https://astralcodexten.substack.com/feed",
    "Noahpinion": "https://noahpinion.substack.com/feed",
    "The Diff": "https://thediff.co/feed",
    "Exponential View": "https://www.exponentialview.co/feed",
    "The Pragmatic Engineer": "https://blog.pragmaticengineer.com/feed",
    "AI Supremacy": "https://aisupremacy.substack.com/feed",
    "The Algorithmic Bridge": "https://thealgorithmicbridge.substack.com/feed",
    "Import AI": "https://importai.substack.com/feed",
    "Machine Learning Engineer": "https://mlengineer.substack.com/feed",
    "Data Science Weekly": "https://datascienceweekly.substack.com/feed"
}

### 3. Feeds Registry

Combine all feeds into one registry with a type prefix for tracking.

In [0]:
ALL_FEEDS = {
    **{
        f"Blog | {name}": url
        for name, url in BLOG_FEEDS.items()
    },
    **{
        f"Newsletter | {name}": url
        for name, url in SUBSTACK_FEEDS.items()
    }
}

print(f"Total RSS feeds: {len(ALL_FEEDS)}")
print(f"Blogs: {len(BLOG_FEEDS)}")
print(f"Newsletters: {len(SUBSTACK_FEEDS)}")

Total RSS feeds: 34
Blogs: 23
Newsletters: 11


### 4. Feeds Validation

Check each feed is reachable and returns valid entries before collection.

In [0]:
import requests
import feedparser

print("\n" + "=" * 100)
print("RSS FEED VALIDATION")
print("=" * 100)

feed_results = []

for feed_name, feed_url in ALL_FEEDS.items():

    try:
        response = requests.get(
            feed_url,
            timeout=20,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        parsed = feedparser.parse(response.content)

        valid = (
            response.status_code == 200
            and len(parsed.entries) > 0
        )

        feed_results.append({
            "feed": feed_name,
            "url": feed_url,
            "http_status": response.status_code,
            "entries": len(parsed.entries),
            "valid": valid
        })

        status = "✅ VALID" if valid else "⚠️ CHECK"

        print(
            f"{status:<10}"
            f"{feed_name:<45}"
            f"HTTP: {response.status_code:<4}"
            f"Entries: {len(parsed.entries)}"
        )

    except Exception as e:

        feed_results.append({
            "feed": feed_name,
            "url": feed_url,
            "http_status": None,
            "entries": 0,
            "valid": False
        })

        print(
            f"❌ ERROR    "
            f"{feed_name:<45}"
            f"{str(e)[:100]}"
        )


# Keep only validated feeds
VALID_FEEDS = {
    result["feed"]: result["url"]
    for result in feed_results
    if result["valid"]
}


print("\n" + "-" * 100)
print(f"Total feeds:             {len(feed_results)}")
print(f"Valid feeds:             {len(VALID_FEEDS)}")
print(f"Invalid feeds:           {len(feed_results) - len(VALID_FEEDS)}")
print(f"Feeds used for collection: {len(VALID_FEEDS)}")


RSS FEED VALIDATION
✅ VALID   Blog | OpenAI                                HTTP: 200 Entries: 1193
✅ VALID   Blog | Google AI                             HTTP: 200 Entries: 20
✅ VALID   Blog | Google DeepMind                       HTTP: 200 Entries: 100
✅ VALID   Blog | Google Research                       HTTP: 200 Entries: 100
✅ VALID   Blog | Hugging Face                          HTTP: 200 Entries: 862
✅ VALID   Blog | Meta Engineering                      HTTP: 200 Entries: 9
✅ VALID   Blog | Apple Machine Learning                HTTP: 200 Entries: 10
✅ VALID   Blog | NVIDIA                                HTTP: 200 Entries: 18
✅ VALID   Blog | NVIDIA Developer                      HTTP: 200 Entries: 100
✅ VALID   Blog | Simon Willison                        HTTP: 200 Entries: 30
✅ VALID   Blog | TechCrunch AI                         HTTP: 200 Entries: 20
✅ VALID   Blog | MIT Technology Review AI              HTTP: 200 Entries: 10
✅ VALID   Blog | The Decoder                      

In [0]:
# Keep only validated RSS feeds for data collection

VALID_FEEDS = {
    result["feed"]: result["url"]
    for result in feed_results
    if result["valid"]
}

print("\n" + "-" * 100)
print(f"Feeds used for collection: {len(VALID_FEEDS)}")


----------------------------------------------------------------------------------------------------
Feeds used for collection: 34


### 5. Topic Detection

Classify each record as AI, Data, or Cloud using keyword scoring.

In [0]:
TOPIC_KEYWORDS = {

    "ai": {
        "strong": [
            "artificial intelligence",
            "machine learning",
            "deep learning",
            "generative ai",
            "large language model",
            "llm",
            "foundation model",
            "computer vision",
            "natural language processing",
            "ai agent",
            "ai agents",
            "reinforcement learning",
            "fine-tuning",
            "multimodal",
            "retrieval augmented generation",
            "rag",
            "transformer"
        ],

        "medium": [
            "neural network",
            "embedding",
            "inference",
            "model training",
            "model evaluation",
            "prompt engineering",
            "reasoning model"
        ]
    },

    "data": {
        "strong": [
            "data engineering",
            "data engineer",
            "data pipeline",
            "data warehouse",
            "data lake",
            "data lakehouse",
            "data architecture",
            "data modeling",
            "etl",
            "elt",
            "apache spark",
            "apache kafka",
            "dbt",
            "snowflake",
            "airflow",
            "data governance",
            "data quality",
            "stream processing",
            "analytics engineering"
        ],

        "medium": [
            "sql",
            "database",
            "big data",
            "data platform",
            "data processing",
            "data analytics",
            "data science",
            "databricks",
            "kafka"
        ]
    },

    "cloud": {
        "strong": [
            "cloud computing",
            "cloud architecture",
            "cloud infrastructure",
            "cloud security",
            "cloud native",
            "kubernetes",
            "serverless",
            "terraform",
            "devops",
            "devsecops",
            "site reliability",
            "sre",
            "microservices",
            "container orchestration"
        ],

        "medium": [
            "aws",
            "azure",
            "google cloud",
            "gcp",
            "docker",
            "container",
            "containers",
            "infrastructure as code",
            "virtual machine"
        ]
    }
}


SOURCE_HINTS = {

    "Databricks": "data",
    "Confluent": "data",
    "Netflix Tech Blog": "data",
    "Uber Engineering": "data",
    "Spotify Engineering": "data",
    "InfoQ Data": "data",
    "InfoQ Big Data": "data",
    "InfoQ Data Engineering": "data",

    "Cloudflare": "cloud",
    "Kubernetes": "cloud",
    "AWS Compute": "cloud",
    "AWS News": "cloud",

    "OpenAI": "ai",
    "Google AI": "ai",
    "Google DeepMind": "ai",
    "Google Research": "ai",
    "Hugging Face": "ai",
    "Microsoft AI": "ai",
    "NVIDIA": "ai",
    "NVIDIA Developer": "ai",
    "Apple Machine Learning": "ai",
    "AI Supremacy": "ai",
    "The Algorithmic Bridge": "ai",
    "Import AI": "ai",
    "Machine Learning Engineer": "ai"
}


def detect_topic(title, description, source):

    title = title.lower()
    description = description.lower()

    scores = {
        "ai": 0,
        "data": 0,
        "cloud": 0
    }

    for topic, groups in TOPIC_KEYWORDS.items():

        for keyword in groups["strong"]:

            if keyword in title:
                scores[topic] += 10

            if keyword in description:
                scores[topic] += 3

        for keyword in groups["medium"]:

            if keyword in title:
                scores[topic] += 5

            if keyword in description:
                scores[topic] += 1


    # Source is only a weak supporting signal
    if source in SOURCE_HINTS:
        scores[SOURCE_HINTS[source]] += 2


    return max(scores, key=scores.get)

### 6. Cleaning Helpers

Utility functions to clean text and normalize dates.

In [0]:
def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text or "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [0]:
def parse_date(entry):
    for field in ["updated", "published"]:
        value = entry.get(field)

        if value:
            try:
                return parsedate_to_datetime(value).astimezone(
                    timezone.utc
                ).isoformat()
            except Exception:
                pass

    for field in ["updated_parsed", "published_parsed"]:
        value = entry.get(field)

        if value:
            try:
                return datetime(
                    *value[:6],
                    tzinfo=timezone.utc
                ).isoformat()
            except Exception:
                pass

    return None

### 7. Record Extraction

Extract, clean, and enrich fields from each feed entry into a standard schema.

In [0]:
# Extract Records from RSS Feeds

import hashlib
from datetime import datetime, timezone
from bs4 import BeautifulSoup


# Detect category based on topic and content
def detect_category(title, description, topic):

    text = f"{title} {description}".lower()

    category_keywords = {

        "ai": {

            "Machine Learning": [
                "machine learning",
                "ml",
                "model training",
                "classification",
                "regression",
                "supervised learning",
                "unsupervised learning"
            ],

            "Deep Learning": [
                "deep learning",
                "neural network",
                "neural networks",
                "cnn",
                "transformer",
                "pytorch",
                "tensorflow"
            ],

            "Natural Language Processing": [
                "nlp",
                "natural language",
                "language model",
                "large language model",
                "llm",
                "text generation",
                "token",
                "embeddings"
            ],

            "Generative AI": [
                "generative ai",
                "genai",
                "generative",
                "chatgpt",
                "gpt",
                "foundation model"
            ],

            "Computer Vision": [
                "computer vision",
                "image recognition",
                "object detection",
                "image classification",
                "image processing"
            ],

            "Reinforcement Learning": [
                "reinforcement learning",
                "rl",
                "reward",
                "agent",
                "policy learning"
            ]
        },

        "data": {

            "Data Engineering": [
                "data engineering",
                "etl",
                "elt",
                "data pipeline",
                "data warehouse",
                "data lake",
                "data processing",
                "data integration"
            ],

            "Data Science": [
                "data science",
                "data analysis",
                "statistics",
                "predictive analytics",
                "statistical analysis"
            ],

            "Databases": [
                "database",
                "sql",
                "mysql",
                "postgresql",
                "mongodb",
                "nosql",
                "database management"
            ],

            "Data Analytics": [
                "data analytics",
                "analytics",
                "business intelligence",
                "bi",
                "dashboard",
                "data visualization"
            ],

            "Big Data": [
                "big data",
                "hadoop",
                "spark",
                "apache spark",
                "distributed computing"
            ],

            "Data Quality": [
                "data quality",
                "data validation",
                "data cleaning",
                "data governance",
                "data management"
            ]
        },

        "cloud": {

            "Cloud Computing": [
                "cloud computing",
                "cloud",
                "cloud infrastructure",
                "cloud services"
            ],

            "DevOps": [
                "devops",
                "ci/cd",
                "continuous integration",
                "continuous deployment",
                "docker"
            ],

            "Kubernetes": [
                "kubernetes",
                "k8s",
                "container orchestration"
            ],

            "Cloud Security": [
                "cloud security",
                "iam",
                "identity",
                "access management",
                "cybersecurity"
            ],

            "Cloud Architecture": [
                "cloud architecture",
                "microservices",
                "serverless",
                "cloud deployment",
                "distributed systems"
            ]
        }
    }

    best_category = None
    best_score = 0

    for category, keywords in category_keywords.get(topic.lower(), {}).items():

        score = sum(
            1 for keyword in keywords
            if keyword in text
        )

        if score > best_score:
            best_score = score
            best_category = category

    # Default category
    if best_category is None:

        if topic.lower() == "ai":
            best_category = "Artificial Intelligence"

        elif topic.lower() == "data":
            best_category = "Data Engineering"

        elif topic.lower() == "cloud":
            best_category = "Cloud Computing"

    return best_category


# Extract multiple keywords related to the content
def extract_keywords(title, description, topic):

    text = f"{title} {description}".lower()

    keyword_groups = {

        "ai": [
            "artificial intelligence",
            "machine learning",
            "deep learning",
            "neural networks",
            "generative ai",
            "llm",
            "large language models",
            "natural language processing",
            "nlp",
            "computer vision",
            "transformers",
            "reinforcement learning",
            "pytorch",
            "tensorflow",
            "gpt",
            "ai models",
            "model training",
            "model evaluation",
            "fine tuning",
            "inference",
            "embeddings",
            "robotics",
            "agents"
        ],

        "data": [
            "data engineering",
            "data science",
            "data analytics",
            "data pipeline",
            "etl",
            "elt",
            "sql",
            "database",
            "data warehouse",
            "data lake",
            "apache spark",
            "big data",
            "data visualization",
            "analytics",
            "data processing",
            "data integration",
            "data modeling",
            "data quality",
            "data management",
            "business intelligence",
            "data governance"
        ],

        "cloud": [
            "cloud computing",
            "cloud infrastructure",
            "aws",
            "azure",
            "google cloud",
            "docker",
            "kubernetes",
            "devops",
            "serverless",
            "microservices",
            "cloud security",
            "cloud architecture",
            "containers",
            "cloud storage",
            "cloud services",
            "infrastructure as code",
            "ci/cd",
            "cloud deployment",
            "distributed systems"
        ]
    }

    keywords = []

    for keyword in keyword_groups.get(topic.lower(), []):

        if keyword in text:
            keywords.append(keyword)

    # Always include the main topic
    if topic.lower() not in keywords:
        keywords.insert(0, topic.lower())

    # Add related keywords if only one keyword was detected
    if len(keywords) < 3:

        keywords.extend(
            keyword_groups.get(topic.lower(), [])[:5]
        )

    # Remove duplicates while preserving order
    keywords = list(dict.fromkeys(keywords))

    return ", ".join(keywords)


def extract_records(feed_name, feed_url):

    records = []

    try:

        response = requests.get(
            feed_url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        response.raise_for_status()

        parsed = feedparser.parse(response.content)

        for entry in parsed.entries:

            # Title
            title = entry.get(
                "title",
                ""
            ).strip()

            # URL
            url = (
                entry.get("link", "")
                or entry.get("id", "")
            ).strip()

            # Description
            description = (
                entry.get("summary", "")
                or entry.get("description", "")
                or ""
            )

            # Remove HTML from description
            description = BeautifulSoup(
                description,
                "html.parser"
            ).get_text(
                " ",
                strip=True
            )

            # Do not copy title into description
            if not description.strip():
                description = "No description available"

            # Skip incomplete records
            if not title or not url:
                continue

            # Published date
            published_date = (
                entry.get("published", "")
                or entry.get("updated", "")
                or ""
            )

            # Topic classification
            topic = detect_topic(
                title,
                description,
                feed_name
            )

            # Keep only project topics
            if topic.lower() not in {
                "ai",
                "data",
                "cloud"
            }:
                continue

            # Normalize topic
            topic = topic.lower()

            # Category
            category = detect_category(
                title,
                description,
                topic
            )

            # Keywords
            keywords = extract_keywords(
                title,
                description,
                topic
            )

            # Content type
            feed_lower = feed_name.lower()

            if "newsletter" in feed_lower:
                content_type = "newsletter"
            else:
                content_type = "article"

            # Deterministic content ID
            content_id = hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()

            # Create record
            # Column order is fixed as requested
            record = {
                "content_id": content_id,
                "title": title,
                "description": description,
                "topic": topic,
                "category": category,
                "list of keywords": keywords,
                "language": "english",
                "source": feed_name,
                "url": url,
                "content_type": content_type,
                "published_date": published_date,
                "last_updated": datetime.now(
                    timezone.utc
                ).isoformat()
            }

            records.append(record)

    except Exception as e:

        print(
            f"Extraction error for {feed_name}: {str(e)}"
        )

    return records

### 8. Collection Run

Collect records from all validated feeds and remove duplicates.

In [0]:
# Collect RSS Records

all_records = []

print("\n" + "=" * 100)
print("RSS COLLECTION REPORT")
print("=" * 100)


for feed_name, feed_url in VALID_FEEDS.items():

    records = extract_records(
        feed_name,
        feed_url
    )

    all_records.extend(records)

    print(
        f"✅ {feed_name:<45} "
        f"{len(records):>4} records"
    )


# Remove duplicate records using content_id
unique_records = list({
    record["content_id"]: record
    for record in all_records
}.values())


print("\n" + "-" * 100)

print(f"Collected records:   {len(all_records)}")
print(f"Unique records:      {len(unique_records)}")
print(
    f"Duplicates removed: "
    f"{len(all_records) - len(unique_records)}"
)

print("-" * 100)

# Show topic distribution
topic_counts = {}

for record in unique_records:

    topic = record["topic"]

    topic_counts[topic] = topic_counts.get(topic, 0) + 1


print("\nTopic Distribution:")

for topic, count in sorted(topic_counts.items()):

    print(f"   {topic:<10} {count:>5} records")


RSS COLLECTION REPORT


/home/spark-fa4efa4d-72ff-42d0-90ef-e0/.ipykernel/75/command-6685736995455133-849012971:343: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  description = BeautifulSoup(


✅ Blog | OpenAI                                 1193 records
✅ Blog | Google AI                                20 records
✅ Blog | Google DeepMind                         100 records
✅ Blog | Google Research                         100 records
✅ Blog | Hugging Face                            862 records
✅ Blog | Meta Engineering                          9 records
✅ Blog | Apple Machine Learning                   10 records
✅ Blog | NVIDIA                                   18 records
✅ Blog | NVIDIA Developer                        100 records
✅ Blog | Simon Willison                           30 records
✅ Blog | TechCrunch AI                            20 records
✅ Blog | MIT Technology Review AI                 10 records
✅ Blog | The Decoder                              10 records
✅ Blog | The Verge AI                             10 records
✅ Blog | Netflix Tech Blog                        10 records
✅ Blog | Spotify Engineering                       5 records
✅ Blog | InfoQ Data     

### 9. Data Quality and Selection

Validate records and select the final 600 records.

In [0]:
# Target number of final records
TARGET_RECORDS = 600

# Required fields for the final dataset
required_fields = [
    "content_id",
    "title",
    "description",
    "topic",
    "category",
    "list of keywords",
    "language",
    "source",
    "url",
    "content_type",
    "published_date",
    "last_updated"
]

# Allowed project topics only
allowed_topics = {"ai", "data", "cloud"}


print("\n" + "=" * 80)
print("DATA QUALITY REPORT")
print("=" * 80)

print(f"\nTotal unique records: {len(unique_records)}")


# Check missing values
print("\nMissing values:")

for field in required_fields:

    missing = sum(
        not record.get(field)
        for record in unique_records
    )

    print(f"{field:<20}: {missing}")


# Check for duplicate content IDs
content_ids = [
    record["content_id"]
    for record in unique_records
]

duplicate_ids = len(content_ids) - len(set(content_ids))

print(f"\nDuplicate content IDs: {duplicate_ids}")


# Check topic distribution
print("\nRecords by topic:")

for topic in ["ai", "data", "cloud"]:

    count = sum(
        record.get("topic") == topic
        for record in unique_records
    )

    print(f"{topic:<10}: {count}")


# Keep only complete records
valid_records = [
    record
    for record in unique_records
    if all(
        record.get(field)
        for field in required_fields
    )
]


# Keep only the three project topics
valid_records = [
    record
    for record in valid_records
    if record["topic"] in allowed_topics
]


print("\n" + "-" * 80)
print(f"Complete valid records: {len(valid_records)}")


# Group records by topic
topic_records = {
    topic: [
        record
        for record in valid_records
        if record["topic"] == topic
    ]
    for topic in ["ai", "data", "cloud"]
}


# Sort each topic deterministically
for topic in topic_records:

    topic_records[topic] = sorted(
        topic_records[topic],
        key=lambda x: (
            x["published_date"] or "",
            x["content_id"]
        ),
        reverse=True
    )


# Flexible round-robin selection
# No fixed quota is enforced
selected_records = []

while len(selected_records) < TARGET_RECORDS:

    records_added = 0

    for topic in ["ai", "data", "cloud"]:

        if topic_records[topic]:

            selected_records.append(
                topic_records[topic].pop(0)
            )

            records_added += 1

            if len(selected_records) >= TARGET_RECORDS:
                break

    # Stop when all available records have been used
    if records_added == 0:
        break


# Final deterministic ordering
selected_records = sorted(
    selected_records,
    key=lambda x: (
        x["published_date"] or "",
        x["content_id"]
    ),
    reverse=True
)


print("\n" + "=" * 80)
print("FINAL DATASET")
print("=" * 80)

print(f"Target records:   {TARGET_RECORDS}")
print(f"Selected records: {len(selected_records)}")


# Final topic distribution
print("\nFinal topic distribution:")

for topic in ["ai", "data", "cloud"]:

    count = sum(
        record["topic"] == topic
        for record in selected_records
    )

    print(f"{topic:<10}: {count}")


# Final topic safety check
invalid_topics = {
    record["topic"]
    for record in selected_records
    if record["topic"] not in allowed_topics
}

if invalid_topics:

    raise ValueError(
        f"Invalid topics found: {invalid_topics}"
    )


# Final duplicate check
final_ids = [
    record["content_id"]
    for record in selected_records
]

if len(final_ids) != len(set(final_ids)):

    raise ValueError(
        "Duplicate content IDs found in the final dataset."
    )


# Final status
print("\n" + "-" * 80)

if len(selected_records) == TARGET_RECORDS:

    print(f" Target reached: {TARGET_RECORDS} records")

else:

    print(
        f" Target not reached. "
        f"Available valid records: {len(selected_records)}"
    )

print(" Only ai, data, and cloud topics are included.")
print(" No fixed topic quota was enforced.")
print(" Selection is deterministic and repeatable.")


DATA QUALITY REPORT

Total unique records: 2825

Missing values:
content_id          : 0
title               : 0
description         : 0
topic               : 0
category            : 0
list of keywords    : 0
language            : 0
source              : 0
url                 : 0
content_type        : 0
published_date      : 0
last_updated        : 0

Duplicate content IDs: 0

Records by topic:
ai        : 2657
data      : 39
cloud     : 129

--------------------------------------------------------------------------------
Complete valid records: 2825

FINAL DATASET
Target records:   600
Selected records: 600

Final topic distribution:
ai        : 432
data      : 39
cloud     : 129

--------------------------------------------------------------------------------
✅ Target reached: 600 records
✅ Only ai, data, and cloud topics are included.
✅ No fixed topic quota was enforced.
✅ Selection is deterministic and repeatable.


### 10. Final Dataset Preview

In [0]:
# Display the complete final dataset

final_df = spark.createDataFrame(selected_records)

display(final_df)

category,content_id,content_type,description,language,last_updated,list of keywords,published_date,source,title,topic,url
Reinforcement Learning,8227954057271af7ea45416a1ed6bee3,article,"We’ve developed Random Network Distillation (RND), a prediction-based method for encouraging reinforcement learning agents to explore their environments through curiosity, which for the first time exceeds average human performance on Montezuma’s Revenge.",english,2026-09-15T20:33:23.562024+00:00,"ai, reinforcement learning, agents","Wed, 31 Oct 2018 07:00:00 GMT",Blog | OpenAI,Reinforcement learning with prediction-based rewards,ai,https://openai.com/index/reinforcement-learning-with-prediction-based-rewards
Reinforcement Learning,e1d6b9833628c22faece92282c2d5286,article,"We’ve trained a model to achieve a new state-of-the-art in mathematical problem solving by rewarding each correct step of reasoning (“process supervision”) instead of simply rewarding the correct final answer (“outcome supervision”). In addition to boosting performance relative to outcome supervision, process supervision also has an important alignment benefit: it directly trains the model to produce a chain-of-thought that is endorsed by humans.",english,2026-09-15T20:33:23.550831+00:00,"ai, artificial intelligence, machine learning, deep learning, neural networks, generative ai","Wed, 31 May 2023 07:00:00 GMT",Blog | OpenAI,Improving mathematical reasoning with process supervision,ai,https://openai.com/index/improving-mathematical-reasoning-with-process-supervision
Artificial Intelligence,7bb5aaedbdc8f0c13755f09e6e5c24e4,article,No description available,english,2026-09-15T20:33:24.547364+00:00,"ai, artificial intelligence, machine learning, deep learning, neural networks, generative ai","Wed, 31 May 2023 00:00:00 GMT",Blog | Hugging Face,Introducing BERTopic Integration with the Hugging Face Hub,ai,https://huggingface.co/blog/bertopic
Natural Language Processing,357e5d2f5c5f2149a699305084f2f80b,article,No description available,english,2026-09-15T20:33:24.547315+00:00,"ai, llm, inference","Wed, 31 May 2023 00:00:00 GMT",Blog | Hugging Face,Introducing the Hugging Face LLM Inference Container for Amazon SageMaker,ai,https://huggingface.co/blog/sagemaker-huggingface-llm
Artificial Intelligence,342444cf66dee91f5a1e13975427acc8,article,No description available,english,2026-09-15T20:33:24.558423+00:00,"ai, artificial intelligence, machine learning, deep learning, neural networks, generative ai","Wed, 31 Mar 2021 00:00:00 GMT",Blog | Hugging Face,Understanding BigBird's Block Sparse Attention,ai,https://huggingface.co/blog/big-bird
Artificial Intelligence,c12250301bc66cabf1c7affe29f26443,article,No description available,english,2026-09-15T20:33:24.532822+00:00,"ai, artificial intelligence, machine learning, deep learning, neural networks, generative ai","Wed, 31 Jul 2024 00:00:00 GMT",Blog | Hugging Face,"Google releases Gemma 2 2B, ShieldGemma and Gemma Scope",ai,https://huggingface.co/blog/gemma-july-update
Natural Language Processing,2f606cb05c0aee02ce41be3c2af86f2b,article,"We’re developing a blueprint for evaluating the risk that a large language model (LLM) could aid someone in creating a biological threat. In an evaluation involving both biology experts and students, we found that GPT-4 provides at most a mild uplift in biological threat creation accuracy. While this uplift is not large enough to be conclusive, our finding is a starting point for continued research and community deliberation.",english,2026-09-15T20:33:23.547142+00:00,"ai, llm, gpt","Wed, 31 Jan 2024 08:00:00 GMT",Blog | OpenAI,Building an early warning system for LLM-aided biological threat creation,ai,https://openai.com/index/building-an-early-warning-system-for-llm-aided-biological-threat-creation
Reinforcement Learning,e873ca19e9b3cf868a306b2378f495d8,article,No description available,english,2026-09-15T20:33:24.538476+00:00,"ai, artificial intelligence, machine learning, deep learning, neural networks, g

## 11. Export Dataset

Download the final dataset as JSON.

In [0]:
import json, base64
from IPython.display import display, HTML

json_filename = 'rss_feeds_600.json'

# Build JSON directly from the in-memory variable
json_str = json.dumps(selected_records, ensure_ascii=False, indent=2)
encoded = base64.b64encode(json_str.encode('utf-8')).decode('utf-8')
size_kb = round(len(json_str.encode('utf-8')) / 1024, 2)

print(f" Records : {len(selected_records)}")
print(f" Size    : {size_kb} KB")
print(" Click below:")

display(HTML(f"""
<a download="{json_filename}"
   href="data:application/json;base64,{encoded}"
   style="display:inline-block;padding:14px 32px;
          background:linear-gradient(135deg,#00A972,#007A52);
          color:white;text-decoration:none;border-radius:8px;
          font-weight:bold;font-family:Arial;font-size:16px;">
   ⬇️ Download {json_filename} ({size_kb} KB)
</a>
"""))

 Records : 600
 Size    : 952.19 KB
 Click below:
